In [ ]:
import subprocess
import time

# Install zstd, which is required for Ollama installation
!sudo apt-get install zstd -y

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Pull the llama3 model if it hasn't been pulled yet
# This command will only download if it's not already present
!ollama pull llama3

# Start ollama serve in the background using nohup to keep it running
# Redirect output to /dev/null to prevent it from cluttering the notebook output
subprocess.Popen(['nohup', 'ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, preexec_fn=subprocess.os.setpgrp)

print("Ollama server started in the background. Waiting a few seconds for initialization...")
time.sleep(10) # Give the server some time to start up
print("Ollama server should now be running.")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 111 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

Ollama server started in the background. Waiting a few seconds for initialization...
Ollama server should now be running.


In [ ]:
%%capture
pip install pypdf

### 1. Load PDF and Extract Text

First, I will load the provided Arabic PDF document and extract all the text content. We'll use the `pypdf` library for this.

In [ ]:
from pypdf import PdfReader

pdf_path = '/content/www.norbook.shop العادات الذرية.pdf'

### Text Extraction using OCR

Since the initial text extraction yielded minimal content, it's highly likely the PDF is image-based. We will now use Optical Character Recognition (OCR) to extract text from the PDF pages. This requires:

1.  **Tesseract OCR Engine**: A powerful open-source OCR engine.
2.  **Tesseract Arabic Language Pack**: To correctly recognize Arabic characters.
3.  **pytesseract**: A Python wrapper for Tesseract.
4.  **Pillow**: Python imaging library.
5.  **PyMuPDF (fitz)**: To robustly convert PDF pages into images, which Tesseract can then process.

In [ ]:
# Install Tesseract OCR engine and Arabic language pack
!sudo apt update
!sudo apt install -y tesseract-ocr
!sudo apt install -y tesseract-ocr-ara

# Install Python libraries
!pip install pytesseract Pillow PyMuPDF

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
111 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as r

In [ ]:
import fitz  # PyMuPDF
import pytesseract
from PIL import Image
import io

# Set the path to the Tesseract executable (if not in PATH)
# This is often the default path on Colab. If you get an error, uncomment and adjust the path.
# pytesseract.pytesseract.tesseract_cmd = r'/usr/bin/tesseract'

def extract_text_from_pdf_with_ocr(pdf_path, start_page=0, end_page=None):
    text_content = []
    try:
        doc = fitz.open(pdf_path)
        total_pages = len(doc)
        print(f"Opened PDF: {pdf_path} with {total_pages} pages.")

        # Adjust end_page if not specified or if it exceeds total pages
        if end_page is None or end_page > total_pages:
            end_page = total_pages

        # Ensure start_page is within valid bounds
        if start_page < 0:
            start_page = 0
        if start_page >= end_page:
            print("Start page is greater than or equal to end page. No pages to process.")
            return ""

        for page_num in range(start_page, end_page):
            page = doc.load_page(page_num)  # Load a page (0-indexed)
            pix = page.get_pixmap(matrix=fitz.Matrix(300/72, 300/72))  # Render page to an image (300 DPI)
            img_bytes = pix.tobytes("png")  # Get image as PNG bytes
            img = Image.open(io.BytesIO(img_bytes)) # Open image with Pillow

            # Perform OCR on the image, specifying Arabic language
            page_text = pytesseract.image_to_string(img, lang='ara')
            text_content.append(page_text)
            print(f"Processed page {page_num + 1}/{total_pages}")
    except Exception as e:
        print(f"Error during OCR: {e}")
    return "\n".join(text_content)

# Apply OCR extraction for the full range (pages 6 to 289)
# Note: Page numbers are 1-indexed in user request, so convert to 0-indexed for `range` and `doc.load_page`
# start_page=5 for 6th page, end_page=289
ocr_text = extract_text_from_pdf_with_ocr(pdf_path, start_page=5, end_page=289)

print(f"\nTotal characters extracted via OCR: {len(ocr_text)}")
print("\nSnippet of OCR extracted text:")
print(ocr_text[:1000])

# Update raw_text to use the OCR'd text for subsequent cleaning steps
raw_text = ocr_text

Opened PDF: /content/www.norbook.shop العادات الذرية.pdf with 291 pages.
Processed page 6/291
Processed page 7/291
Processed page 8/291
Processed page 9/291
Processed page 10/291
Processed page 11/291
Processed page 12/291
Processed page 13/291
Processed page 14/291
Processed page 15/291
Processed page 16/291
Processed page 17/291
Processed page 18/291
Processed page 19/291
Processed page 20/291
Processed page 21/291
Processed page 22/291
Processed page 23/291
Processed page 24/291
Processed page 25/291
Processed page 26/291
Processed page 27/291
Processed page 28/291
Processed page 29/291
Processed page 30/291
Processed page 31/291
Processed page 32/291
Processed page 33/291
Processed page 34/291
Processed page 35/291
Processed page 36/291
Processed page 37/291
Processed page 38/291
Processed page 39/291
Processed page 40/291
Processed page 41/291
Processed page 42/291
Processed page 43/291
Processed page 44/291
Processed page 45/291
Processed page 46/291
Processed page 47/291
Process

### Clean the text

In [ ]:
import re
import unicodedata
from dataclasses import dataclass, field
from typing import Optional, Set
import io
from PIL import Image
import fitz
import pytesseract

# Helper functions for Arabic text preprocessing
def normalize_arabic(text: str, keep_hamza: bool = False) -> str:
    text = re.sub(r'[إأٱآ]', 'ا', text)  # Normalize alif variants
    if not keep_hamza:
        text = re.sub(r'[ؤئ]', 'ء', text)  # Normalize hamza on waw/yeh to hamza
    text = re.sub(r'ة', 'ه', text)  # Normalize taa marbuta to haa
    text = re.sub(r'ى', 'ي', text)  # Normalize alif maqsura to yaa
    # Remove tashkeel (diacritics)
    text = re.sub(r'[ًٌٍَُِّْ]', '', text)
    return text

def remove_arabic_noise(
    text: str,
    remove_html: bool = True,
    remove_urls: bool = True,
    remove_mentions: bool = True,
    remove_hashtags: bool = True,
    remove_emails: bool = True,
    remove_tatweel: bool = True,
) -> str:
    if remove_html:
        text = re.sub(r'<.*?>', '', text) # Remove HTML tags
    if remove_urls:
        text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text) # Remove URLs
    if remove_mentions:
        text = re.sub(r'@[\w_]+', '', text) # Remove mentions
    if remove_hashtags:
        text = re.sub(r'#\w+', '', text) # Remove hashtags
    if remove_emails:
        text = re.sub(r'\S*@\S*\s?', '', text) # Remove emails
    if remove_tatweel:
        text = re.sub(r'ـ', '', text) # Remove tatweel
    return text

def handle_emojis(text: str, mode: str = 'remove') -> str:
    if mode == 'remove':
        # More robust emoji removal
        emoji_pattern = re.compile(
            "["
            "\U0001F600-\U0001F64F"  # emoticons
            "\U0001F300-\U0001F5FF"  # symbols & pictographs
            "\U0001F680-\U0001F6FF"  # transport & map symbols
            "\U0001F1E0-\U0001F1FF"  # flags (iOS)
            "\U00002702-\U000027B0"
            "\U000024C2-\U0001F251"
            "]+", flags=re.UNICODE
        )
        return emoji_pattern.sub(r'', text)
    # 'description' and 'placeholder' modes would require a more complex library/mapping
    return text

def correct_arabic_text(text: str, custom_corrections: dict = None) -> str:
    # This is a placeholder. A full spelling corrector requires a large dictionary and algorithm.
    # For now, it only applies custom corrections.
    if custom_corrections:
        for wrong, correct in custom_corrections.items():
            text = text.replace(wrong, correct)
    return text

def normalize_elongated_words(text: str, max_repeat: int = 2) -> str:
    # Remove elongation (e.g., 'ووو' -> 'وو')
    text = re.sub(r'(.)\1{' + str(max_repeat) + ',}', r'\1' * max_repeat, text)
    return text

def remove_punctuations(text: str) -> str:
    arabic_punctuations = '''`÷×؛<>_()*&^%][ـ،/:";?.,'{}~''' # Corrected: properly closed triple quote and added some common punctuations
    # Common Arabic punctuation characters
    text = re.sub(r'[' + re.escape(arabic_punctuations) + ']', '', text)
    # English punctuations
    text = re.sub(r'[!"#$%&()*+,-./:;<=>?@[\]^_`{|}~]', '', text)
    return text

def handle_numbers(text: str, mode: str = 'remove') -> str:
    if mode == 'remove':
        return re.sub(r'\d+', '', text) # Remove digits
    elif mode == 'normalize':
        # Convert Eastern Arabic numerals to Western Arabic numerals if present
        text = text.replace('٠', '0').replace('١', '1').replace('٢', '2').replace('٣', '3').replace('٤', '4').replace('٥', '5').replace('٦', '6').replace('٧', '7').replace('٨', '8').replace('٩', '9')
        return text
    return text

def filter_non_arabic(text: str, keep_english: bool = False, keep_numbers: bool = False) -> str:
    # Allow Arabic, spaces, and optionally English letters and numbers
    pattern = r'[^؀-\u06FF\s'
    if keep_english:
        pattern += 'a-zA-Z'
    if keep_numbers:
        pattern += '0-9'
    pattern += ']'
    return re.sub(pattern, '', text)

def tokenize_arabic(text: str, method: str = 'whitespace') -> list:
    if method == 'whitespace':
        return text.split()
    # Other methods (e.g., Arabic-specific tokenizers) would require libraries like NLTK or Farasa
    return text.split()

def remove_stopwords(tokens: list, custom_stopwords: Set[str] = None, min_token_length: int = 2) -> list:
    # This is a very basic list. A more comprehensive list would be needed.
    default_arabic_stopwords = set([
        'من', 'في', 'إلى', 'على', 'عن', 'مع', 'أو', 'و', 'لا', 'يا', 'قد', 'إن',
        'هذا', 'هذه', 'هو', 'هي', 'هم', 'هن', 'أن', 'إلا', 'لو', 'ما', 'هنا'
    ])
    if custom_stopwords:
        stopwords = default_arabic_stopwords.union(custom_stopwords)
    else:
        stopwords = default_arabic_stopwords
    return [token for token in tokens if token not in stopwords and len(token) >= min_token_length]

def stem_arabic_tokens(tokens: list) -> list:
    # This is a placeholder. Arabic stemming is complex and requires specialized libraries (e.g., ISRIStemmer)
    # For this project, a simple passthrough or a custom rule-based stemmer could be implemented if a library is not desired.
    return tokens


@dataclass
class ArabicPreprocessingConfig:
    """Configuration flags for the full preprocessing pipeline."""
    # Normalization
    normalize:            bool = True
    keep_hamza:           bool = False
    # Noise removal
    remove_html:          bool = True
    remove_urls:          bool = True
    remove_mentions:      bool = True
    remove_hashtags:      bool = True
    remove_emails:        bool = True
    remove_tatweel:       bool = True
    # Punctuation
    remove_punctuations:  bool = True
    # Spelling
    correct_spelling:     bool = True
    custom_corrections:   dict = field(default_factory=dict)
    # Numbers
    number_mode:          str  = 'remove'   # 'remove' | 'normalize' | 'keep'
    # Emojis
    emoji_mode:           str  = 'remove'   # 'remove' | 'description' | 'placeholder' | 'keep'
    # Elongation
    normalize_elongation: bool = True
    max_repeat:           int  = 2
    # Language filtering
    filter_non_arabic:    bool = False
    keep_english:         bool = False
    keep_numbers:         bool = False
    # Tokenization
    tokenize:             bool = True
    tokenize_method:      str  = 'whitespace'
    # Stopwords
    remove_stopwords:     bool = True
    custom_stopwords:     Set[str] = field(default_factory=set)
    min_token_length:     int  = 2
    # Stemming
    stem:                 bool = False


def preprocess_arabic(
    text: str,
    config: ArabicPreprocessingConfig = None,
    return_tokens: bool = False,
):
    """
    Full Arabic text preprocessing pipeline.

    Args:
        text:          Raw input text
        config:        ArabicPreprocessingConfig (uses defaults if None)
        return_tokens: If True, returns list of tokens; else returns joined string

    Returns:
        Preprocessed string or token list
    """
    cfg = config or ArabicPreprocessingConfig()

    # Step 1 — Normalize
    if cfg.normalize:
        text = normalize_arabic(text, keep_hamza=cfg.keep_hamza)

    # Step 2 — Remove noise
    text = remove_arabic_noise(
        text,
        remove_html=cfg.remove_html,
        remove_urls=cfg.remove_urls,
        remove_mentions=cfg.remove_mentions,
        remove_hashtags=cfg.remove_hashtags,
        remove_emails=cfg.remove_emails,
        remove_tatweel=cfg.remove_tatweel,
    )

    # Step 3 — Handle emojis
    if cfg.emoji_mode != 'keep':
        text = handle_emojis(text, mode=cfg.emoji_mode)

    # Step 4 — Spelling correction
    if cfg.correct_spelling:
        text = correct_arabic_text(text, cfg.custom_corrections)

    # Step 5 — Elongation
    if cfg.normalize_elongation:
        text = normalize_elongated_words(text, max_repeat=cfg.max_repeat)

    # Step 6 — Punctuation
    if cfg.remove_punctuations:
        text = remove_punctuations(text)

    # Step 7 — Numbers
    if cfg.number_mode != 'keep':
        text = handle_numbers(text, mode=cfg.number_mode)

    # Step 8 — Language filter
    if cfg.filter_non_arabic:
        text = filter_non_arabic(
            text,
            keep_english=cfg.keep_english,
            keep_numbers=cfg.keep_numbers,
        )

    # Step 9 — Tokenize
    if cfg.tokenize or cfg.remove_stopwords or cfg.stem:
        tokens = tokenize_arabic(text, method=cfg.tokenize_method)
    else:
        return text

    # Step 10 — Stopwords
    if cfg.remove_stopwords:
        tokens = remove_stopwords(
            tokens,
            custom_stopwords=cfg.custom_stopwords,
            min_token_length=cfg.min_token_length,
        )

    # Step 11 — Stemming
    if cfg.stem:
        tokens = stem_arabic_tokens(tokens)

    return tokens if return_tokens else ' '.join(tokens)

# Custom config — keep English, stem, return tokens
custom_cfg = ArabicPreprocessingConfig(
    filter_non_arabic=True,
    keep_english=True,
    stem=True,
)

# Apply the preprocessing to the OCR-extracted raw_text
cleaning_config = ArabicPreprocessingConfig(
    normalize=True,
    remove_html=True,
    remove_urls=True,
    remove_mentions=True,
    remove_hashtags=True,
    remove_emails=True,
    remove_tatweel=True,
    remove_punctuations=True,
    correct_spelling=False, # Set to False for now, as a full spelling corrector is complex and not yet implemented robustly
    number_mode='remove',
    emoji_mode='remove',
    normalize_elongation=True,
    filter_non_arabic=True,
    keep_english=False,
    keep_numbers=False,
    tokenize=False, # We want the cleaned text as a string for now, not tokens
    remove_stopwords=True,
    stem=False
)

cleaned_text = preprocess_arabic(raw_text, config=cleaning_config, return_tokens=False)

print(f"Original OCR text length: {len(raw_text)}")
print(f"Cleaned text length: {len(cleaned_text)}")
print("\nSnippet of cleaned text:")
print(cleaned_text[:1000])


Original OCR text length: 327689
Cleaned text length: 293153

Snippet of cleaned text:
اليوم الاخير عامي الدراسي الثاني بالمدرسه الثانويه صربت علي وجهي بمضرب بيسبول اذ بينما كان زميل لي يطوح مضربه بطول ذراعه انفلت المضرب يديه وطار نحوي ليصطدم بي بين عيني مباشره ليس لدي اي ذكري لحظه الارتطام ضرب المضرب وجهي بقوه شديده الي درجه انه سحق انفي وشوهه كما تسبب ارتطام انسجه دماغي الرخوه بقوه بالجزء الداخلي جمجمتي وعلي الفور تورم راسي وفي جزء الثانيه تهشم انفي وتعرضت جمجمتي لكسور مواضع متعدده وتهشم حين فتحت عيني رايت اشخاصا يحدقون بي ويهرعون لتقديم يد المساعده نظرت الي الاسفل ولاحظت بقعه حمراء علي ثيابي خلع زميل لي قميصه وناولني اياه فاستخدمته سد تيار الدماء المندفع انفي المهشم كنت مصدوما ومرتبكاء ولم اكن اعي مدي خطوره اصابتي وضع مدرسي ذراعه اسفل كتفي وشرعنا السير الي مكتب الممرضه فعبرنا الملعب ونزلنا تلاء ثم دخلنا الي مبني المدرسه ساندتني اياد عده حتي اسير معتدلا كنا نسير ببطء وتءده ولم يكن احد يدرك ان كل دقيقه مهمه حين وصلنا الي مكتب الممرضه سالتني عددا الاسءله اي عام نحن؟ اجبتها العام لكثنا 

### 7. Advanced Chunking and Retrieval Strategies

To implement more advanced chunking and retrieval strategies, we'll need additional libraries for embeddings, vector stores, and specialized text splitters. We'll also define helper functions for sentence splitting and token counting which are essential for many of these methods.

In [ ]:
# Install necessary libraries
!pip install -q langchain langchain-community faiss-cpu sentence-transformers accelerate
!pip install -q langchain-text-splitters chromadb tiktoken pypdf langchain-core
!pip install -U langchain-ollama
!pip install langchain-classic # For ParentDocumentRetriever and related components

import re
import numpy as np
from typing import List
import tiktoken

def split_sentences(text: str) -> List[str]:
    """Splits Arabic text into sentences using common punctuation."""
    # Arabic sentence ending punctuation: . ; ! ?
    # Also handle newlines as potential sentence breaks
    sentences = re.split(r'(?<=[.؛!?])\s*|\n\s*', text)
    # Filter out empty strings that might result from splitting
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences

def count_tokens(text: str) -> int:
    """Counts tokens using cl100k_base tokenizer (common for OpenAI models)."""
    # Fallback to a simpler token count if tiktoken fails or for non-English
    try:
        encoding = tiktoken.get_encoding("cl100k_base")
        return len(encoding.encode(text))
    except Exception:
        return len(text.split())

print("Required libraries installed and helper functions defined.")

Required libraries installed and helper functions defined.


### 7.1  Recursive Text Splitting

Now that the Arabic text is cleaned, we will divide it into smaller, manageable chunks. We'll use `RecursiveCharacterTextSplitter` to maintain contextual coherence within each chunk. This method recursively tries to split by different characters to find splits that work well.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Define chunking parameters
chunk_size = 1000
chunk_overlap = 100

# Initialize the RecursiveCharacterTextSplitter
# We'll use a list of common Arabic separators and newline characters
# The order matters: it tries to split by the first separator, then the second if the first isn't present, etc.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    length_function=len,
    # Arabic-specific separators (common punctuation, spaces, and newlines)
    separators=[
        "\n\n", # Paragraphs
        "\n",   # Newlines
        " ",    # Spaces
        ".",    # Sentence ends
        "؟",    # Arabic question mark
        "!",
        "،",    # Arabic comma
        "،",    # Arabic comma
        "ـ",    # Tatweel
        "-",    # Hyphen
        "",     # Fallback to character-level splitting
    ],
    is_separator_regex=False # Set to True if separators contain regex patterns
)

# Create document chunks from the cleaned text
doc_chunks = text_splitter.create_documents([cleaned_text])

print(f"Number of document chunks: {len(doc_chunks)}")
print(f"First chunk snippet: {doc_chunks[0].page_content[:200]}...")
print(f"Length of first chunk: {len(doc_chunks[0].page_content)} characters")
print(f"Second chunk snippet: {doc_chunks[1].page_content[:200]}...")
print(f"Length of second chunk: {len(doc_chunks[1].page_content)} characters")

Number of document chunks: 326
First chunk snippet: اليوم الاخير عامي الدراسي الثاني بالمدرسه الثانويه صربت علي وجهي بمضرب بيسبول اذ بينما كان زميل لي يطوح مضربه بطول ذراعه انفلت المضرب يديه وطار نحوي ليصطدم بي بين عيني مباشره ليس لدي اي ذكري لحظه الار...
Length of first chunk: 1000 characters
Second chunk snippet: العام لكثنا الواقع كنا العام رءيس الولايات المتحده الامريكيه؟ قلت بيل كلينتون وكانت الاجابه الصحيحه جورج دبليو بوش اسم والدتك؟ لامم تلعثمت الاجابه ومرت عشر ثوان ثم قلت باتي متجاهلا حقيقه انني استغرقت ...
Length of second chunk: 998 characters


### 7.2 Semantic Chunking (Basic)

Semantic chunking aims to create chunks based on the semantic similarity of sentences. Sentences that are semantically similar are grouped together, ensuring contextual coherence. We use `sentence-transformers` to embed sentences and `util.cos_sim` to calculate similarity. A fixed threshold is used to determine chunk boundaries.

In [ ]:
from sentence_transformers import SentenceTransformer, util

class SemanticChunker:
    """Basic semantic chunking using sentence embeddings with a fixed threshold."""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2", threshold: float = 0.6):
        self.model = SentenceTransformer(model_name)
        self.threshold = threshold

    def chunk(self, text: str) -> List[str]:
        sentences = split_sentences(text)
        if len(sentences) <= 1:
            return sentences

        # Embed all sentences
        embeddings = self.model.encode(sentences, convert_to_tensor=True)
        chunks = []
        current_chunk = [sentences[0]]

        for i in range(1, len(sentences)):
            # Cosine similarity between previous and current sentence
            sim = util.cos_sim(embeddings[i-1], embeddings[i]).item()
            if sim < self.threshold:
                # Break here, start new chunk
                chunks.append(" ".join(current_chunk))
                current_chunk = [sentences[i]]
            else:
                current_chunk.append(sentences[i])
        if current_chunk:
            chunks.append(" ".join(current_chunk))
        return chunks

semantic_chunker = SemanticChunker(threshold=0.5)
semantic_chunks = semantic_chunker.chunk(cleaned_text)
print(f"Basic semantic chunking produced {len(semantic_chunks)} chunks.")
print("\nFirst 3 semantic chunks:")
for i, chunk in enumerate(semantic_chunks[:3]):
    print(f"Chunk {i+1}: {chunk[:200]}...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Basic semantic chunking produced 1 chunks.

First 3 semantic chunks:
Chunk 1: اليوم الاخير عامي الدراسي الثاني بالمدرسه الثانويه صربت علي وجهي بمضرب بيسبول اذ بينما كان زميل لي يطوح مضربه بطول ذراعه انفلت المضرب يديه وطار نحوي ليصطدم بي بين عيني مباشره ليس لدي اي ذكري لحظه الار...


### 7.3 Adaptive Semantic Chunking

Adaptive semantic chunking dynamically determines chunk boundaries based on the statistical distribution of sentence similarities. This allows for more flexible and context-aware chunk sizes, breaking where semantic shifts are more pronounced.

In [ ]:
class AdaptiveSemanticChunker:
    """Fully adaptive semantic chunking using sentence embeddings."""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2", method: str = "iqr", multiplier: float = 1.5):
        """
        method: 'iqr', 'std', 'percentile'
        multiplier: for IQR (1.5) or number of standard deviations (1.0)
        """
        self.model = SentenceTransformer(model_name)
        self.method = method
        self.multiplier = multiplier

    def chunk(self, text: str) -> List[str]:
        sentences = split_sentences(text)   # use your NLTK sentence splitter
        if len(sentences) <= 1:
            return sentences

        # Get embeddings for all sentences
        embeddings = self.model.encode(sentences, convert_to_tensor=True)

        # Compute cosine similarities between consecutive sentences
        similarities = []
        for i in range(1, len(sentences)):
            sim = util.cos_sim(embeddings[i-1], embeddings[i]).item()
            similarities.append(sim)

        # Determine breakpoint threshold adaptively based on similarity distribution
        if self.method == "iqr":
            q1 = np.percentile(similarities, 25)
            q3 = np.percentile(similarities, 75)
            iqr = q3 - q1
            threshold = q1 - self.multiplier * iqr   # break when similarity is low
        elif self.method == "std":
            mean = np.mean(similarities)
            std = np.std(similarities)
            threshold = mean - self.multiplier * std
        elif self.method == "percentile":
            threshold = np.percentile(similarities, 25)  # lowest 25% are breakpoints
        else:
            raise ValueError("method must be 'iqr', 'std', or 'percentile'")

        # Build chunks
        chunks = []
        current_chunk = [sentences[0]]
        for i in range(1, len(sentences)):
            sim = similarities[i-1]
            if sim < threshold:
                chunks.append(" ".join(current_chunk))
                current_chunk = [sentences[i]]
            else:
                current_chunk.append(sentences[i])
        if current_chunk:
            chunks.append(" ".join(current_chunk))
        return chunks

adaptive_chunker = AdaptiveSemanticChunker(method="percentile")
adaptive_chunks = adaptive_chunker.chunk(cleaned_text)
print(f"Adaptive semantic chunking produced {len(adaptive_chunks)} chunks.")
print("\nFirst 3 adaptive semantic chunks:")
for i, chunk in enumerate(adaptive_chunks[:3]):
    print(f"Chunk {i+1}: {chunk[:200]}...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Adaptive semantic chunking produced 1 chunks.

First 3 adaptive semantic chunks:
Chunk 1: اليوم الاخير عامي الدراسي الثاني بالمدرسه الثانويه صربت علي وجهي بمضرب بيسبول اذ بينما كان زميل لي يطوح مضربه بطول ذراعه انفلت المضرب يديه وطار نحوي ليصطدم بي بين عيني مباشره ليس لدي اي ذكري لحظه الار...


### 7.4 Document-Based Chunking (Markdown Headers)

Document-based chunking leverages the inherent structure of documents, such as markdown headers, to create logical chunks. This method is particularly useful for structured documents, ensuring that sections and sub-sections are treated as coherent units.

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

sample_text_markdown = """
# مقدمة إلى العادات الذرية
العادات الذرية هي تغييرات صغيرة تحدث فرقًا كبيرًا.

## ما هي العادة الذرية؟
هي عادة صغيرة ومراكمة. غالبًا ما تكون غير ملحوظة.

### كيف تعمل العادات الذرية؟
تعمل من خلال مبدأ التراكم.

## قانون العادات الأربعة

1.  **اجعلها واضحة:** اجعل إشارة العادة واضحة.
2.  **اجعلها جذابة:** اجعلها مغرية.
3.  **اجعلها سهلة:** اجعلها بسيطة قدر الإمكان.
4.  **اجعلها مُرضية:** اجعلها مجزية.

### المرحلة الأولى: اجعلها واضحة

### المرحلة الثانية: اجعلها جذابة

### المرحلة الثالثة: اجعلها سهلة

### المرحلة الرابعة: اجعلها مُرضية

## الخلاصة
التغييرات الصغيرة تؤدي إلى نتائج عظيمة مع الوقت.
"""

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_docs = markdown_splitter.split_text(sample_text_markdown)
print(f"Document-based chunking produced {len(md_docs)} chunks.")
print("\nFirst 3 Markdown chunks with metadata:")
for i, doc in enumerate(md_docs[:3]):
    print(f"Chunk {i+1} (Header 1: {doc.metadata.get('Header 1', 'N/A')}, Header 2: {doc.metadata.get('Header 2', 'N/A')}, Header 3: {doc.metadata.get('Header 3', 'N/A')}):")
    print(f"{doc.page_content[:200]}...")

Document-based chunking produced 5 chunks.

First 3 Markdown chunks with metadata:
Chunk 1 (Header 1: مقدمة إلى العادات الذرية, Header 2: N/A, Header 3: N/A):
العادات الذرية هي تغييرات صغيرة تحدث فرقًا كبيرًا....
Chunk 2 (Header 1: مقدمة إلى العادات الذرية, Header 2: ما هي العادة الذرية؟, Header 3: N/A):
هي عادة صغيرة ومراكمة. غالبًا ما تكون غير ملحوظة....
Chunk 3 (Header 1: مقدمة إلى العادات الذرية, Header 2: ما هي العادة الذرية؟, Header 3: كيف تعمل العادات الذرية؟):
تعمل من خلال مبدأ التراكم....


### 7.5 Agentic Chunking (LLM as Editor)

Agentic chunking leverages an LLM to intelligently decide where to break text into chunks. This method aims to produce highly coherent chunks by allowing a language model to understand the context and make informed decisions about logical boundaries. This technique can be computationally intensive due to repeated LLM calls, so it's best demonstrated with a smaller text sample.

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

# Initialize a local LLM (e.g., using Ollama or another LangChain compatible LLM)
# Using a small, fast model is recommended for agentic chunking due to API calls.
llm = ChatOllama(
    model="llama3", # Replace with your preferred local LLM model
    temperature=0.0
)

def agentic_chunking(text: str, llm_model: ChatOllama, max_sentences_per_batch: int = 10) -> List[str]:
    """
    Uses an LLM to decide chunk boundaries sentence by sentence.
    To reduce cost and latency, it processes sentences in small batches.
    """
    sentences = split_sentences(text)
    if not sentences:
        return []

    # The prompt for the LLM to decide on chunk breaks
    prompt_template = ChatPromptTemplate.from_messages([
        ("system", "You are an expert at finding logical breaks in text. Given two adjacent sentences, decide if a new chunk should start after the first sentence. Answer only YES or NO."),
        ("human", "Sentence A: {sent_a}\nSentence B: {sent_b}\nShould we break here? Answer only YES or NO.")
    ])
    chain = prompt_template | llm

    chunks = []
    current_chunk = [sentences[0]]

    # Process sentences in batches to optimize LLM calls (though this simplified version checks sentence by sentence)
    for i in range(len(sentences)-1):
        response = chain.invoke({"sent_a": sentences[i], "sent_b": sentences[i+1]})
        decision = response.content.strip().upper()
        if decision == "YES":
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i+1]]
        else:
            current_chunk.append(sentences[i+1])
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    return chunks

# Using a smaller part of the cleaned_text for demonstration due to LLM call costs
sample_for_agentic = cleaned_text[5000:6000]
agentic_chunks = agentic_chunking(sample_for_agentic, llm)
print(f"Agentic chunking produced {len(agentic_chunks)} chunks for the sample.")
print("\nFirst 3 agentic chunks:")
for i, chunk in enumerate(agentic_chunks[:3]):
    print(f"Chunk {i+1}: {chunk[:200]}...")

Agentic chunking produced 1 chunks for the sample.

First 3 agentic chunks:
Chunk 1: ابعه وفي نظر شخص كرس الوقت والمجهود لهذه الرياضه كان الاستبعاد امرا مهيءا واذكر جيدا اليوم الذي حدث فيه فقد جلست سيارتي واخذت ادير مءشر محطات الراديو وانا ابحث قانطا اغنيه شانها ان تحسن حالتي المزاجيه...


### 7.6 Parent-Document Retrieval (Hierarchical)

Parent-document retrieval is a strategy where small, highly granular "child" chunks are used for retrieval (e.g., in a vector store), but when a relevant child chunk is found, a larger, more contextually rich "parent" document (or a larger segment of it) is returned to the LLM. This provides a balance between precise retrieval and sufficient context.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_classic.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.schema import Document

# Splitters for parent and child documents
# Parent splitter creates larger chunks for context
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50, length_function=count_tokens)
# Child splitter creates smaller chunks for more granular retrieval
child_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20, length_function=count_tokens)

# Vectorstore and storage
# Using a SentenceTransformer model for embeddings
embeddor = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# Using Chroma as the vectorstore for similarity search
vectorstore = Chroma(embedding_function=embeddor)
# Using InMemoryStore for storing parent documents
store = InMemoryStore()

# Initialize the ParentDocumentRetriever
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    search_kwargs={"k": 4} # Retrieve top 4 child chunks
)

# Create a single document from the cleaned text for the retriever
doc = Document(page_content=cleaned_text, metadata={"source": "arabic_pdf"})

# Add the document to the retriever
retriever.add_documents([doc])
print("Parent-document retriever created and document added.")

# Example query for Parent-Document Retrieval
query_parent_doc = "ما هي العادات الذرية؟"
retrieved_parent_docs = retriever.invoke(query_parent_doc)

print(f"\nNumber of documents retrieved by Parent-Document Retriever: {len(retrieved_parent_docs)}")
if retrieved_parent_docs:
    print("\nSnippet of first parent-retrieved document:")
    print(retrieved_parent_docs[0].page_content[:500])
    print(f"Source: {retrieved_parent_docs[0].metadata.get('source', 'N/A')}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Parent-document retriever created and document added.

Number of documents retrieved by Parent-Document Retriever: 4

Snippet of first parent-retrieved document:
بالتحفيز كي تكون صاحب قوام ممشوقء خطط لجلسات اليوجا وادفع ثمنها مقدما واذا كنت تشعر بالحماسه بشان العمل الذي تريد ان تبداه ابعث بريدا الكتر ونيا الي راءد اعمال تحترمه ورتب معه مكالمه استشاريه وحين ياتي وقت الفعل يكون المخرج الوحيد الغاء الاجتماع وهذا يتطلب جهدا ويكلف مالا ان ادوات الالتزام تزيد احتمالات فعلك للشيء الصواب المستقبل طريق جعل العادات السيءه صعبه الحاضر ومع ذلك يمكننا فعل افضل ذلك اذ يمكننا جعل العادات الحسنه حتميه والعادات السيءه مستحيله كيف تءتمت العاده ولا تفكر فيها مجددا ولد جون 
Source: arabic_pdf


### 7.7 Knowledge Graph Representation and Evaluation

In [ ]:
from langchain_core.documents import Document
import networkx as nx
import re

# Assuming llm_evaluation is defined in an earlier cell for consistency
llm_model = llm_evaluation

# Use the cleaned text from the PDF, not hardcoded examples
sample_texts = [cleaned_text]

# Use the evaluation questions defined earlier, which are in Arabic and relevant to the PDF
questions = evaluation_questions

# Create document chunks
chunks = [Document(page_content=text, metadata={}) for text in sample_texts]

def extract_simple_entities_relations(text):
    prompt = f"""
    استخرج الكيانات والعلاقات من هذا النص.
    قم بتنسيق كل علاقة بالشكل التالي: كيان1 -[علاقة]-> كيان2

    النص: {text}

    أمثلة:
    - العادات الذرية -[هي]-> تغييرات صغيرة
    - التغييرات الصغيرة -[تؤدي إلى]-> نتائج عظيمة
    """

    try:
        response = llm_model.invoke(prompt)
        return response.content
    except Exception as e:
        print(f"LLM error: {e}")
        return ""

In [ ]:
#Building Knowledge Graph

# Initialize knowledge graph
G = nx.DiGraph()

# Ensure llm_model is defined in this scope for graph extraction
llm_model = llm_evaluation

# Re-initialize the Parent-Document Retriever and its components for the Knowledge Graph RAG
# Splitters for parent and child documents (assuming they are still defined in kernel, or re-define if needed)
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50, length_function=count_tokens)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20, length_function=count_tokens)

# Vectorstore and storage
embeddor = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# Using a temporary client to ensure a fresh in-memory Chroma instance
vectorstore = Chroma(embedding_function=embeddor)
store = InMemoryStore()

# Create a single document from the cleaned text for the retriever
doc = Document(page_content=cleaned_text, metadata={"source": "arabic_pdf"})

# Initialize the ParentDocumentRetriever
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    search_kwargs={"k": 4}
)

# Add the document to the retriever
retriever.add_documents([doc])
print("Parent-document retriever re-initialized and document added for Knowledge Graph RAG.")


# Use the recursively split chunks for graph population
# This provides smaller, more manageable contexts for LLM entity/relation extraction
chunks_for_graph = doc_chunks

# Extract relationships from each chunk
for i, chunk in enumerate(chunks_for_graph):
    # Get entity-relation triples from LLM
    triples_text = extract_simple_entities_relations(chunk.page_content)

    # Parse the response
    lines = [line.strip() for line in triples_text.split("\n") if line.strip()]

    for line in lines:
        # Look for pattern: Entity1 -[Relation]-> Entity2
        match = re.search(r"(.+?)\s*-\[(.+?)\]->\s*(.+)", line)
        if match:
            entity1, relation, entity2 = match.groups()
            entity1 = entity1.strip()
            entity2 = entity2.strip()
            relation = relation.strip()

            # Add to graph
            G.add_edge(entity1, entity2, relation=relation)


def graph_retrieve(query_entity, depth=2):
    """Retrieve relevant information from knowledge graph"""

    # Find entities that contain the query term (case-insensitive)
    matching_entities = [node for node in G.nodes()
                        if query_entity.lower() in node.lower()]

    if not matching_entities:
        print(f"No entities found matching '{query_entity}'")
        return []

    print(f"Found matching entities: {matching_entities}")

    # Get subgraph around matching entities
    relevant_nodes = set()
    for entity in matching_entities:
        # Get nodes within specified depth
        try:
            nodes = nx.single_source_shortest_path_length(G, entity, cutoff=depth)
            relevant_nodes.update(nodes.keys())
        except:
            pass

    # Extract relationships from relevant subgraph
    subgraph_facts = []
    for src, dst in G.edges():
        if src in relevant_nodes or dst in relevant_nodes:
            relation = G[src][dst]['relation']
            subgraph_facts.append(f"{src} -[{relation}]-> {dst}")

    return subgraph_facts


def knowledge_graph_rag(question: str):

    semantic_docs = retriever.invoke(question)
    semantic_context = "\n".join([d.page_content for d in semantic_docs])

    words = question.split()
    key_entities = [word for word in words if len(word) > 3 and word.isalpha()]

    graph_facts = []
    for entity in key_entities:
        facts = graph_retrieve(entity, depth=2)
        graph_facts.extend(facts)

    graph_context = "\n".join(set(graph_facts))  # Remove duplicates


    combined_context = f"""SEMANTIC CONTEXT:{semantic_context}
     KNOWLEDGE GRAPH FACTS:{graph_context}"""

    prompt = f"""Use the provided context and knowledge graph facts to answer the question comprehensively.
    {combined_context}
    Question: {question}
    Answer: """

    try:
        answer = llm_model.invoke(prompt)
        return answer, semantic_context, graph_context
    except Exception as e:
        return f"Error generating answer: {e}", semantic_context, graph_context



for question in questions:
   answer, semantic_ctx, graph_ctx = knowledge_graph_rag(question)

print(f"\nQUESTION: {question}")
print(f"ANSWER: {answer}")
print(f"\nSemantic Context Used:\n{semantic_ctx}")
print(f"\nGraph Facts Used:\n{graph_ctx}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Parent-document retriever re-initialized and document added for Knowledge Graph RAG.
Found matching entities: ['5. العادات البسيطه', 'العادات ذي الاربع خطوات', '4. العادات ذي الاربع خطوات', '* العادات الذرية', '4. **العادات** (Habits)', '10. **العادات الحسنه** (Good habits)', '4. العادات المهمله القذره', '1. العادات الذريه (Habits)', '2. العادات الذريه (Habits)', '3. العادات الذريه (Habits)', 'مصطلح العادات الذريه (The concept of habits)', '4. العادات الروتينيه (Routine habits)', '5. العادات الروتينيه (Routine habits)', '7. العادات الذريه (Habits)', '1. **العادات الذريه**', '**العادات الذريه**', '6. **العادات الذريه**', '11. **العادات السيءه**', '**الصعب بناء العادات الحسنه**', '3. العادات السيئة (Bad habits)', '8. العادات غير الصحية (Unhealthy habits)', '9. العادات المرتكزه علي النتاءج (Habits Based on Results)', '10. العادات المرتكزه علي الهويه (Habits Based on Identity)', '5. العادات', 'اهميه العادات', '3. العادات', '6. العادات', '13. العادات', 'اهميه العادات (again)', '3. اهميه الع

### 7.8 Unified Retrieval Pipeline and Evaluation Setup

Now that we have explored various chunking and retrieval strategies, the next crucial step is to build a unified retrieval pipeline for each method and connect it to the same LLM using a consistent prompt and evaluation questions. This setup will allow us to compare the effectiveness of different retrieval techniques in terms of relevance, completeness, accuracy, and context richness for Arabic text.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

# Initialize the same LLM for all evaluations
# Using a local LLM for consistency and reproducibility
llm_evaluation = ChatOllama(
    model="llama3", # Ensure this is the same LLM used across all comparisons
    temperature=0.1 # Keep temperature low for consistent outputs
)

# Define a consistent prompt template for all retrieval methods
rqa_prompt = ChatPromptTemplate.from_messages([
    ("system", "أنت مساعد ذكاء اصطناعي خبير مهمتك هي الإجابة على الأسئلة بناءً فقط على السياق المقدم. إذا لم تكن الإجابة موجودة في السياق، فاذكر أنك لا تعرف."),
    ("human", "السياق: {context}\n\nالسؤال: {question}")
])

# Define example evaluation questions
evaluation_questions = [
    "ما هي العادات الذرية ولماذا هي مهمة؟",
    "كيف يمكن للمرء أن يبني عادات جيدة وفقًا للمؤلف؟",
    "ما هي أهمية البيئة في تكوين العادات؟",
    "هل الكتاب يقدم استراتيجيات محددة لتغيير العادات السيئة؟ اذكر بعض الأمثلة."
]

print("LLM, unified prompt, and evaluation questions initialized for comparison.")

LLM, unified prompt, and evaluation questions initialized for comparison.


### 7.9 Evaluation: Recursive Character Text Splitting

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("Evaluating Recursive Character Text Splitting...")

# 1. Create a vector store from the recursive chunks
# Using a SentenceTransformer model for embeddings
embeddor = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
recursive_vectorstore = Chroma.from_documents(documents=doc_chunks, embedding=embeddor)
recursive_retriever = recursive_vectorstore.as_retriever(search_kwargs={"k": 4})

# 2. Build the RAG chain
recursive_rag_chain = (
    {"context": recursive_retriever, "question": RunnablePassthrough()}
    | rqa_prompt
    | llm_evaluation
    | StrOutputParser()
)

# 3. Run evaluation questions
print("\n--- Responses for Recursive Character Text Splitting ---")
for i, question in enumerate(evaluation_questions):
    print(f"\nQuestion {i+1}: {question}")
    response = recursive_rag_chain.invoke(question)
    print(f"Answer: {response}")

# Clean up the vectorstore to free up resources if no longer needed
recursive_vectorstore.delete_collection()
print("\nRecursive vector store collection deleted.")

Evaluating Recursive Character Text Splitting...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Responses for Recursive Character Text Splitting ---

Question 1: ما هي العادات الذرية ولماذا هي مهمة؟
Answer: Based on the context, I can infer that the "عادات ذريه" (Habits) refer to the habits or behaviors that are ingrained in an individual's daily life. These habits can be good or bad and can have a significant impact on one's success or failure.

According to the text, these habits can be changed through strategies that cover various aspects of life, such as health, wealth, productivity, relationships, etc. The importance of changing these habits lies in their ability to shape one's destiny and future.

In the context of the British cycling team, for example, changing their habits led to significant improvements in their performance and ultimately, they were able to win Olympic medals.

Similarly, in the context of business, changing one's habits can lead to increased productivity, efficiency, and profitability. The text suggests that by adopting good habits, individuals can

### 7.10 Evaluation: Basic Semantic Chunking

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document

print("Evaluating Basic Semantic Chunking...")

# 1. Create a list of Document objects from semantic_chunks
# Each semantic chunk becomes a document. Add some metadata if available, e.g., 'source'.
semantic_docs = [Document(page_content=chunk, metadata={"source": "arabic_pdf_semantic_chunk"}) for chunk in semantic_chunks]

# 2. Create a vector store from the semantic chunks
embeddor = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
semantic_vectorstore = Chroma.from_documents(documents=semantic_docs, embedding=embeddor)
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k": 4})

# 3. Build the RAG chain
semantic_rag_chain = (
    {"context": semantic_retriever, "question": RunnablePassthrough()}
    | rqa_prompt
    | llm_evaluation
    | StrOutputParser()
)

# 4. Run evaluation questions
print("\n--- Responses for Basic Semantic Chunking ---")
for i, question in enumerate(evaluation_questions):
    print(f"\nQuestion {i+1}: {question}")
    response = semantic_rag_chain.invoke(question)
    print(f"Answer: {response}")

# Clean up the vectorstore to free up resources
semantic_vectorstore.delete_collection()
print("\nBasic semantic vector store collection deleted.")

Evaluating Basic Semantic Chunking...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Responses for Basic Semantic Chunking ---

Question 1: ما هي العادات الذرية ولماذا هي مهمة؟
Answer: Based on the provided context, it seems that the documents are discussing genetics and neuroscience. However, I don't see any specific information about "العادات الذرية" (genetic habits) or why they might be important.

As a knowledgeable AI assistant, I must admit that I don't have enough information to provide an answer to this question based on the provided context. If you could provide more context or clarify what you mean by "العادات الذرية", I'd be happy to try and help you further!

Question 2: كيف يمكن للمرء أن يبني عادات جيدة وفقًا للمؤلف؟
Answer: Based on the provided context, it seems that the author is discussing the importance of good habits and how to build them. According to the text, the key is not just to read books (هويتك ليس الهدف ان تقرا كتاباء بل), but rather to focus on one's work and avoid distractions (يهدر العاملون اي وقت الالتفات كي). Additionally, it seems

### 7.11 Evaluation: Adaptive Semantic Chunking

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document

print("Evaluating Adaptive Semantic Chunking...")

# 1. Create a list of Document objects from adaptive_chunks
adaptive_docs = [Document(page_content=chunk, metadata={"source": "arabic_pdf_adaptive_semantic_chunk"}) for chunk in adaptive_chunks]

# 2. Create a vector store from the adaptive semantic chunks
embeddor = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
adaptive_semantic_vectorstore = Chroma.from_documents(documents=adaptive_docs, embedding=embeddor)
adaptive_semantic_retriever = adaptive_semantic_vectorstore.as_retriever(search_kwargs={"k": 4})

# 3. Build the RAG chain
adaptive_semantic_rag_chain = (
    {"context": adaptive_semantic_retriever, "question": RunnablePassthrough()}
    | rqa_prompt
    | llm_evaluation
    | StrOutputParser()
)

# 4. Run evaluation questions
print("\n--- Responses for Adaptive Semantic Chunking ---")
for i, question in enumerate(evaluation_questions):
    print(f"\nQuestion {i+1}: {question}")
    response = adaptive_semantic_rag_chain.invoke(question)
    print(f"Answer: {response}")

# Clean up the vectorstore to free up resources
adaptive_semantic_vectorstore.delete_collection()
print("\nAdaptive semantic vector store collection deleted.")

Evaluating Adaptive Semantic Chunking...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Responses for Adaptive Semantic Chunking ---

Question 1: ما هي العادات الذرية ولماذا هي مهمة؟
Answer: What a fascinating topic!

The habits you're referring to are called "Habits of Mind" (HoM). They were first introduced by Art Costa and Bena Kallick in their book "Assessment, Evaluation, Selection, and Validity: A Guide for Educators" (2000).

Habits of Mind are a set of 16 habits that help individuals develop a growth mindset, think critically, and become more effective learners. These habits include:

1. Perseverance
2. Thinking about your thinking (metacognition)
3. Open-mindedness
4. Taking responsible risks
5. Finding humor
6. Presenting and interpreting your work
7. Striving for accuracy
8. Thinking critically
9. Creating, imagining, innovating
10. Embracing the unknown
11. Being a reflective thinker
12. Seeking connections
13. Developing self-awareness
14. Taking pride in your work
15. Showing respect and empathy towards others
16. Staying curious

These habits are impor

### 7.12 Evaluation: Document-Based Chunking (Markdown Headers)

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("Evaluating Document-Based Chunking (Markdown Headers)...")

# 1. Create a vector store from the markdown chunks
embeddor = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
markdown_vectorstore = Chroma.from_documents(documents=md_docs, embedding=embeddor)
markdown_retriever = markdown_vectorstore.as_retriever(search_kwargs={"k": 4})

# 2. Build the RAG chain
markdown_rag_chain = (
    {"context": markdown_retriever, "question": RunnablePassthrough()}
    | rqa_prompt
    | llm_evaluation
    | StrOutputParser()
)

# 3. Run evaluation questions
print("\n--- Responses for Document-Based Chunking ---")
for i, question in enumerate(evaluation_questions):
    print(f"\nQuestion {i+1}: {question}")
    response = markdown_rag_chain.invoke(question)
    print(f"Answer: {response}")

# Clean up the vectorstore to free up resources
markdown_vectorstore.delete_collection()
print("\nMarkdown vector store collection deleted.")

Evaluating Document-Based Chunking (Markdown Headers)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Responses for Document-Based Chunking ---

Question 1: ما هي العادات الذرية ولماذا هي مهمة؟
Answer: Based on the provided context, I can answer your question.

According to the text, "العادات الذرية" (Habits) are small changes that happen unnoticed. They are described as "تغييرات صغيرة تحدث فرقًا كبيرًا" (small changes that make a big difference).

As for why they are important, it is mentioned in the context that "التغييرات الصغيرة تؤدي إلى نتائج عظيمة مع الوقت" (small changes lead to great results over time). Additionally, it is stated that "تعمل من خلال مبدأ التراكم" (it works through the principle of accumulation), implying that small habits can add up to significant outcomes.

In summary, habits are small changes that happen unnoticed and are important because they can lead to great results over time.

Question 2: كيف يمكن للمرء أن يبني عادات جيدة وفقًا للمؤلف؟
Answer: According to the context, it seems that the author suggests building good habits through small changes that 

### 7.13 Evaluation: Agentic Chunking

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document

print("Evaluating Agentic Chunking...")

# 1. Create a list of Document objects from agentic_chunks
agentic_docs = [Document(page_content=chunk, metadata={"source": "arabic_pdf_agentic_chunk"}) for chunk in agentic_chunks]

# 2. Create a vector store from the agentic chunks
embeddor = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
agentic_vectorstore = Chroma.from_documents(documents=agentic_docs, embedding=embeddor)
agentic_retriever = agentic_vectorstore.as_retriever(search_kwargs={"k": 4})

# 3. Build the RAG chain
agentic_rag_chain = (
    {"context": agentic_retriever, "question": RunnablePassthrough()}
    | rqa_prompt
    | llm_evaluation
    | StrOutputParser()
)

# 4. Run evaluation questions
print("\n--- Responses for Agentic Chunking ---")
# Note: Agentic chunking was performed on a small sample, so answers might be limited
for i, question in enumerate(evaluation_questions):
    print(f"\nQuestion {i+1}: {question}")
    response = agentic_rag_chain.invoke(question)
    print(f"Answer: {response}")

# Clean up the vectorstore to free up resources
agentic_vectorstore.delete_collection()
print("\nAgentic vector store collection deleted.")

Evaluating Agentic Chunking...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Responses for Agentic Chunking ---

Question 1: ما هي العادات الذرية ولماذا هي مهمة؟
Answer: Based on the context, it seems that the habits you are referring to are the simple habits that you discovered during your time at Denisson University. You mentioned that these habits were "simple" and had a "powerful" impact.

From what I can gather from the text, these habits seem to be related to your athletic performance and overall well-being. You mentioned that you became a university athlete and were able to overcome some challenges during your studies.

As for why these habits are important, it seems that they played a crucial role in your personal growth and development. You mentioned that you were able to make significant progress and achieve your goals through the power of these simple habits.

In summary, the habits you are referring to seem to be related to your athletic performance and overall well-being, and their importance lies in their ability to help you overcome challeng

### 7.14 Evaluation: Parent-Document Retrieval

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_classic.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.schema import Document

print("Evaluating Parent-Document Retrieval...")

# Re-initialize the Parent-Document Retriever and its components to ensure a fresh state
# Splitters for parent and child documents
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50, length_function=count_tokens)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20, length_function=count_tokens)

# Vectorstore and storage
embeddor = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(embedding_function=embeddor) # This will create a new, distinct Chroma collection
store = InMemoryStore()

# Initialize the ParentDocumentRetriever
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    search_kwargs={"k": 4}
)

# Create a single document from the cleaned text for the retriever
doc = Document(page_content=cleaned_text, metadata={"source": "arabic_pdf"})

# Add the document to the retriever
retriever.add_documents([doc])
print("Parent-document retriever re-initialized and document added for evaluation.")

# 2. Build the RAG chain using the (re-initialized) retriever
parent_doc_rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | rqa_prompt
    | llm_evaluation
    | StrOutputParser()
)

# 3. Run evaluation questions
print("\n--- Responses for Parent-Document Retrieval ---")
for i, question in enumerate(evaluation_questions):
    print(f"\nQuestion {i+1}: {question}")
    response = parent_doc_rag_chain.invoke(question)
    print(f"Answer: {response}")

# Clean up its specific vectorstore after evaluation (optional, but good practice for isolated tests)
vectorstore.delete_collection()
print("\nParent-Document Retriever's vector store collection deleted.")


Evaluating Parent-Document Retrieval...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Parent-document retriever re-initialized and document added for evaluation.

--- Responses for Parent-Document Retrieval ---

Question 1: ما هي العادات الذرية ولماذا هي مهمة؟
Answer: Based on the context, it seems that the author is discussing habits and their importance. The text mentions "العادات السيءه" (bad habits) and "العادات الحسنه" (good habits), as well as the concept of creating new habits through repetition and consistency.

According to the text, good habits can be created by making them automatic through repetition, and this process is referred to as the "تاييد طويل المدي" (long-term consolidation) or "قانون هيب" (Hebb's law). This refers to the strengthening of neural connections in the brain through repeated behavior.

The text also emphasizes the importance of creating good habits by stating that they can lead to positive outcomes, such as improved physical and mental health. It suggests that creating good habits requires effort and consistency, but it is worth it in th